In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.decomposition import PCA
from sklearn.feature_selection import SelectKBest, mutual_info_classif
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report, confusion_matrix

from imblearn.under_sampling import RandomUnderSampler
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from catboost import CatBoostClassifier

import joblib
import warnings
warnings.filterwarnings('ignore')

print("=" * 60)
print("HYBRID ENSEMBLE MODEL FOR WSN INTRUSION DETECTION")
print("=" * 60)

# ============================================
# 1. LOAD DATA
# ============================================

data = pd.read_csv(r"dataset\WSN-DS.csv")
data.columns = data.columns.str.strip()

data.rename(columns={
    'who CH': 'who_CH',
    'Expaned Energy': 'Expanded_Energy',
    'Attack type': 'Attack_type'
}, inplace=True)

if 'id' in data.columns:
    data.drop('id', axis=1, inplace=True)

X = data.drop('Attack_type', axis=1)
y = data['Attack_type']

print("\nOriginal class distribution:")
print(y.value_counts())

# ============================================
# 2. HANDLE IMBALANCE
# ============================================

undersampler = RandomUnderSampler(random_state=42)
X_resampled, y_resampled = undersampler.fit_resample(X, y)

# ✅ FIX 1: LABEL ENCODING (for XGBoost)
label_encoder = LabelEncoder()
y_encoded = label_encoder.fit_transform(y_resampled)

# ============================================
# 3. FEATURE OPTIMIZATION
# ============================================

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_resampled)

pca = PCA(n_components=0.95)
X_pca = pca.fit_transform(X_scaled)

mi_selector = SelectKBest(score_func=mutual_info_classif, k='all')
mi_selector.fit(X_pca, y_encoded)

mi_scores = mi_selector.scores_
n_top_features = min(12, X_pca.shape[1])
top_features_idx = np.argsort(mi_scores)[-n_top_features:]

X_optimized = X_pca[:, top_features_idx]

# ============================================
# 4. TRAIN-TEST SPLIT
# ============================================

X_train, X_test, y_train, y_test = train_test_split(
    X_optimized, y_encoded,
    test_size=0.2,
    random_state=42,
    stratify=y_encoded
)

# ============================================
# 5. TRAIN MODELS
# ============================================

models = {}
performance = {}

# --- Random Forest ---
rf_model = RandomForestClassifier(
    n_estimators=200,
    max_depth=15,
    min_samples_split=5,
    random_state=42,
    n_jobs=-1
)
rf_model.fit(X_train, y_train)
rf_pred = rf_model.predict(X_test)

# --- XGBoost ---
xgb_model = XGBClassifier(
    n_estimators=150,
    learning_rate=0.1,
    max_depth=8,
    random_state=42,
    eval_metric='mlogloss'
)
xgb_model.fit(X_train, y_train)
xgb_pred = xgb_model.predict(X_test)

# --- CatBoost ---
cat_model = CatBoostClassifier(
    iterations=200,
    learning_rate=0.1,
    depth=8,
    random_seed=42,
    verbose=False
)
cat_model.fit(X_train, y_train)
cat_pred = cat_model.predict(X_test)

models = {
    'Random Forest': rf_model,
    'XGBoost': xgb_model,
    'CatBoost': cat_model
}

# ============================================
# 6. METRICS FUNCTION
# ============================================

def get_metrics(y_true, y_pred):
    return {
        'accuracy': accuracy_score(y_true, y_pred),
        'precision': precision_score(y_true, y_pred, average='weighted'),
        'recall': recall_score(y_true, y_pred, average='weighted'),
        'f1': f1_score(y_true, y_pred, average='weighted')
    }

performance['Random Forest'] = get_metrics(y_test, rf_pred)
performance['XGBoost'] = get_metrics(y_test, xgb_pred)
performance['CatBoost'] = get_metrics(y_test, cat_pred)

# ============================================
# 7. ENSEMBLE (FIXED)
# ============================================

# Calculate weights
total_f1 = sum(p['f1'] for p in performance.values())
weights = {m: performance[m]['f1'] / total_f1 for m in performance}

# ✅ FIX 2: Robust Voting Function
def weighted_voting(models, weights, X):
    preds = [np.ravel(model.predict(X)) for model in models.values()]
    final_pred = []

    for i in range(len(X)):
        votes = {}
        for model_name, pred in zip(models.keys(), preds):
            cls = int(pred[i])  # ✅ ensures scalar (fixes error)
            votes[cls] = votes.get(cls, 0) + weights[model_name]
        final_pred.append(max(votes, key=votes.get))

    return np.array(final_pred)

ensemble_pred = weighted_voting(models, weights, X_test)
ensemble_metrics = get_metrics(y_test, ensemble_pred)

# ============================================
# 8. RESULTS
# ============================================

print("\nPerformance Comparison:")
for m in performance:
    print(f"{m}: {performance[m]}")

print(f"\nHybrid Ensemble: {ensemble_metrics}")

# ============================================
# 9. CONFUSION MATRIX
# ============================================

def plot_cm(y_true, y_pred, title):
    cm = confusion_matrix(y_true, y_pred)
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
    plt.title(title)
    plt.xlabel("Predicted")
    plt.ylabel("Actual")
    plt.show()

plot_cm(y_test, rf_pred, "Random Forest")
plot_cm(y_test, xgb_pred, "XGBoost")
plot_cm(y_test, cat_pred, "CatBoost")
plot_cm(y_test, ensemble_pred, "Hybrid Ensemble")

# ============================================
# 10. CLASSIFICATION REPORT (DECODED)
# ============================================

y_test_labels = label_encoder.inverse_transform(y_test)

print("\nRANDOM FOREST REPORT")
print(classification_report(y_test_labels, label_encoder.inverse_transform(rf_pred)))

print("\nXGBOOST REPORT")
print(classification_report(y_test_labels, label_encoder.inverse_transform(xgb_pred)))

print("\nCATBOOST REPORT")
print(classification_report(y_test_labels, label_encoder.inverse_transform(cat_pred)))

print("\nENSEMBLE REPORT")
print(classification_report(y_test_labels, label_encoder.inverse_transform(ensemble_pred)))

# ============================================
# 11. SAVE MODELS
# ============================================

# joblib.dump(rf_model, 'models/rf_model.pkl')
# joblib.dump(xgb_model, 'models/xgb_model.pkl')
# joblib.dump(cat_model, 'models/cat_model.pkl')
# joblib.dump(label_encoder, 'models/label_encoder.pkl')

# joblib.dump({
#     'models': models,
#     'weights': weights,
#     'scaler': scaler,
#     'pca': pca,
#     'top_features_idx': top_features_idx
# }, 'models/hybrid_ensemble.pkl')

# print("\n✅ ALL DONE SUCCESSFULLY")

HYBRID ENSEMBLE MODEL FOR WSN INTRUSION DETECTION

Original class distribution:
Attack_type
Normal       340066
Grayhole      14596
Blackhole     10049
TDMA           6638
Flooding       3312
Name: count, dtype: int64


TypeError: unhashable type: 'numpy.ndarray'